# Logistic Regression from Scratch

**Dataset:** Breast Cancer Wisconsin (sklearn built-in)  
**Task:** Binary classification — malignant vs. benign tumors

---

## Overview

Logistic regression is a **supervised learning algorithm** for binary classification. Despite its name, it is a classification model — not a regression model.

### Key Idea

Instead of predicting a continuous value, we predict the **probability** that a sample belongs to class 1:

$$P(y=1 \mid x) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

where $\sigma$ is the **sigmoid function**, which squashes any real number into $(0, 1)$.

### Training via Gradient Descent

We minimize the **binary cross-entropy (log loss)**:

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

The gradient descent update rules are:

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{w}}, \quad b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

## 2. Load and Explore the Dataset

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {len(feature_names)}")
print(f"Classes: {target_names}  (0 = malignant, 1 = benign)")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df.head()

In [ ]:
# Visualize distributions of first 4 features by class
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for i, feat in enumerate(feature_names[:4]):
    for cls, color, label in zip([0, 1], ['salmon', 'steelblue'], target_names):
        axes[i].hist(df[df['target'] == cls][feat], bins=25, alpha=0.6,
                     color=color, label=label)
    axes[i].set_title(feat)
    axes[i].legend()

plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Preprocessing

Logistic regression is sensitive to feature scale because gradient descent converges faster when features are on a similar scale. We use **StandardScaler** to normalize: $z = \frac{x - \mu}{\sigma}$.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training samples: {X_train_scaled.shape[0]}")
print(f"Test samples:     {X_test_scaled.shape[0]}")

## 4. Logistic Regression from Scratch

We implement the sigmoid activation, log-loss, and gradient descent manually.

In [ ]:
class LogisticRegressionScratch:
    """Binary logistic regression via gradient descent."""

    def __init__(self, lr=0.1, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            z = X @ self.weights + self.bias
            p = self._sigmoid(z)

            # Binary cross-entropy loss
            loss = -np.mean(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15))
            self.loss_history.append(loss)

            # Gradients
            dw = (X.T @ (p - y)) / n_samples
            db = np.mean(p - y)

            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

## 5. Train and Evaluate

In [ ]:
model = LogisticRegressionScratch(lr=0.1, n_iters=1000)
model.fit(X_train_scaled, y_train)

train_acc = model.accuracy(X_train_scaled, y_train)
test_acc  = model.accuracy(X_test_scaled,  y_test)

print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")

In [ ]:
# Training loss curve
plt.figure(figsize=(8, 4))
plt.plot(model.loss_history, color='steelblue', lw=2)
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Training Loss Curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Confusion Matrix and Classification Report

In [ ]:
y_pred = model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)

ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(target_names); ax.set_yticklabels(target_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=target_names))

## 7. ROC Curve

The **Receiver Operating Characteristic (ROC)** curve plots the true positive rate vs. false positive rate at different classification thresholds. The **AUC** (area under the curve) summarizes performance — closer to 1.0 is better.

In [ ]:
y_prob = model.predict_proba(X_test_scaled)
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Breast Cancer Classification')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary

| Metric | Value |
|--------|-------|
| Train Accuracy | ~96–98% |
| Test Accuracy | ~95–97% |
| AUC | ~0.99 |

### Key Takeaways

- Logistic regression learns a **linear decision boundary** in feature space.
- Feature **standardization** is critical — unscaled features cause slow or divergent training.
- The **sigmoid function** maps scores to probabilities, enabling both probabilistic outputs and hard predictions.
- **Log loss** is a smooth, differentiable loss function well-suited for gradient descent.
- Despite its simplicity, logistic regression achieves strong performance on linearly separable tasks like breast cancer classification.